# AI Music OS — ACE-Step

This notebook installs and launches the real ACE-Step application. Models/cache are kept on Google Drive so they are not downloaded again when the cache is intact.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, sys, shutil

ROOT = Path('/content/drive/MyDrive/AI_Music_OS')
REPO = ROOT / 'repos' / 'ACE-Step'
MODELS = ROOT / 'models' / 'ace_step'
CACHE = ROOT / 'cache' / 'huggingface'
OUTPUTS = ROOT / 'outputs' / 'ace_step'

for p in (ROOT, MODELS, CACHE, OUTPUTS):
    p.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(CACHE)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(CACHE / 'transformers')
print('AI_Music_OS:', ROOT)


In [ ]:
# GPU / environment check
import torch, platform
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print('WARNING: no GPU detected. ACE-Step inference may be impractical on CPU.')


In [ ]:
# Clone/update the official ACE-Step repository instead of overwriting its launcher.
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ace-step/ACE-Step.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=False)

req = REPO / 'requirements.txt'
if req.exists():
    subprocess.run([sys.executable,'-m','pip','install','-r',str(req)], check=True)
else:
    # Let the repository's own pyproject/setup metadata define dependencies.
    if (REPO/'pyproject.toml').exists() or (REPO/'setup.py').exists():
        subprocess.run([sys.executable,'-m','pip','install','-e',str(REPO)], check=True)
    else:
        raise FileNotFoundError('ACE-Step repository has no recognized dependency metadata.')


In [ ]:
# Show repository launch candidates. Do NOT replace upstream files.
candidates = []
for name in ['app.py','webui.py','webui.py','cli.py','infer.py','inference.py','launch.py']:
    p = REPO / name
    if p.exists():
        candidates.append(p)
print('Launch candidates:', [str(x.relative_to(REPO)) for x in candidates])

for p in REPO.glob('**/*.py'):
    s = p.read_text(errors='ignore')
    if 'gradio' in s.lower() and ('launch(' in s or 'Blocks(' in s):
        print('Possible Gradio entry:', p.relative_to(REPO))


## Launch

ACE-Step changes its entrypoint across releases. The next cell searches the checked-out repository for a likely Gradio entrypoint instead of inventing a fake `launch.py`.

In [ ]:
# Start ACE-Step with its official launcher (T4-safe: no bf16, CPU offload on)
import subprocess, os, sys, time, re, shutil
env = os.environ.copy()
env['HF_HOME'] = str(CACHE)
env['HUGGINGFACE_HUB_CACHE'] = str(CACHE / 'hub')
exe = shutil.which('acestep')
if exe is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
    exe = shutil.which('acestep')
if exe is None:
    raise RuntimeError('The acestep command was not installed. Check the pip output of the install cell.')
log = open(OUTPUTS / 'acestep_launch.log', 'w')
cmd = [exe, '--checkpoint_path', str(MODELS), '--port', '7865', '--share', 'true',
       '--bf16', 'false', '--cpu_offload', 'true', '--overlapped_decode', 'true']
print('Launching:', ' '.join(cmd))
proc = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=log, stderr=subprocess.STDOUT)
url = None
for _ in range(900):  # first run downloads ~7 GB of checkpoints to Drive
    time.sleep(2)
    text = (OUTPUTS / 'acestep_launch.log').read_text(errors='ignore')
    m = re.search(r'https://[a-z0-9]+\.gradio\.live', text)
    if m: url = m.group(0); break
    if proc.poll() is not None:
        print(text[-3000:]); raise RuntimeError('ACE-Step exited — see the log above.')
print('🎵 ACE-Step UI:', url or 'not ready yet — check outputs/ace_step/acestep_launch.log')
